# `earthlib` API Walkthrough

This notebook showcases the fundamental API design patterns and basic use patterns of the `earthlib` package.

## Background: all pixels are mixed

`earthlib` provides a curated spectral library for global land cover mapping. It features a collection of high quality spectral observations collected from open data portals and a series of modeled vegetation spectra that capture the optical dynamics of canopy vegetation.

Modeled vegetation spectra are included because most earth observing sensors — spaceborne or airborne sensors — measure at spatial resolutions much coarser than the scale of individual leaves. Satellite observations measure larger areas that include multiple organisms within a pixel — be they multiple trees, shrubs, or grasses — and canopy reflectance is optically distinct from leaf-level reflectance, driven by the scattering of photons among leaves in a canopy.

Quantifying these mixture dynamics — the inclusion of not only multiple organisms within a pixel but also multiple land cover types — is the purpose of spectral mixture analysis. The fundamental premise is that all pixels are mixed. Linear spectral unmixing provides an analytical framework for estimating the underlying fractions of land cover types within a pixel, given a representative reference of land cover observations.

## Reference spectra: an atomic representation of land surface components

- many land cover representations
- from multiple data providers
- capturing as much global variance in components
- optimized for maximum entropy
- narrowing in on the fundamental expressions land cover types

## Multi-sensor support: a new approach to analysis-ready data

- multiple earth observing sensors measure optical data at different wavelengths
- land cover data, sampled at fine native resolution, can be resampled to any of these sensors
- by using the same reference endmembers, you could run unmixing on multi sensor data, then fuse the results
- this provides a practical approach to sensor fusion that does not rely on sensor-level harmonization, but on data-driven harmonization

These points are all to be made in the paper and probably not here. But I'm saving it here for now!

In [1]:
%load_ext autoreload
%autoreload 2
    
# packages
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt

import earthlib
from earthlib import library

# plotting defaults
mpl.style.use('ggplot')

## Reference library

The `earthlib` reference spectral library are exposed via the `earthlib.library` instance.

This is an instance of athe `earthlib.endmembers.Spectra` class, which provides abstractions for working with spectral library data. The three core components of the spectral library are:

- The reflectance data (the `.data` attribute), which represent the raw spectral estimates
- The library metadata (the `.metadata` attribute), which describe the attributes of each spectra
- The sensor specification (the `.sensor` attribute), which describes the band centers and other observation parameters

The `.data` attribute is simply a numpy ndarray

In [2]:
print(f"Number of reference spectra: {len(library)}")
print(f"Array shape: {library.data.shape}")
print(f"Data type: {library.data.dtype}")

Number of reference spectra: 317
Array shape: (317, 180)
Data type: float32


The `.metatdata` attribute is a pandas DataFrame

In [3]:
library.metadata.head(3)

,NAME,SOURCE,LEVEL_1,LEVEL_2,LEVEL_3,LEVEL_4,LAT,LON,NOTES
0,FS15R_FS4281,icraf-isric-soil-database,pervious,bare,soil,measured,-32.850000,137.533333,none
1,FS15R_FS4297,icraf-isric-soil-database,pervious,bare,soil,measured,-35.350000,144.850000,none
2,FS15R_FS4308,icraf-isric-soil-database,pervious,bare,soil,measured,-29.016667,152.033333,none


Metadata are hierarchically organized by "level"

In [4]:
# level 1: is water absorbed by this land cover type?
print("Level 1 classes:", library.metadata['LEVEL_1'].unique())

# level 2: core land cover types
print("Level 2 classes:", library.metadata['LEVEL_2'].unique())

# level 3: specific observation types
print("Level 3 classes:", library.metadata['LEVEL_3'].unique())

# level 4: measurement type
print("Level 4 classes:", library.metadata['LEVEL_4'].unique())

Level 1 classes: ['pervious' 'impervious']
Level 2 classes: ['bare' 'burn' 'npv' 'urban' 'vegetation']
Level 3 classes: ['soil' 'gravel' 'char' 'npv' 'bark' 'wood' 'litter' 'asphalt'
 'comp_shingle' 'concrete_tile' 'driveway' 'glass' 'manhole' 'metal'
 'other' 'paint' 'road' 'sidewalk' 'tile' 'wood_shingle' 'canopy']
Level 4 classes: ['measured' 'simulated']


The library can be indexed using this metadata

In [5]:
npv = library.metadata['LEVEL_2'] == "npv"
npv_library = library[npv]

print(f"Array shape: {npv_library.data.shape}")
npv_library.metadata.head(3)

Array shape: (42, 180)


,NAME,SOURCE,LEVEL_1,LEVEL_2,LEVEL_3,LEVEL_4,LAT,LON,NOTES
0,MCON_Tower_CWD1_Canopy,specevo,pervious,npv,npv,measured,37.066990,-119.195500,hyspiri simulation spectra
1,MCON_Tower_CWD2_Canopy,specevo,pervious,npv,npv,measured,37.066950,-119.195540,hyspiri simulation spectra
2,SJER_Plot361_NPV_T015,asd,pervious,npv,npv,measured,37.084376,-119.733826,hyspiri simulation spectra


Spectra can be indexed with arrays

In [6]:
random_idxs = [100, 200, 300]
sub_library = library[random_idxs]

print(f"Array shape: {sub_library.data.shape}")
print(f"Data type: {sub_library.data.dtype}")
sub_library.metadata.head()

Array shape: (3, 180)
Data type: float32


,NAME,SOURCE,LEVEL_1,LEVEL_2,LEVEL_3,LEVEL_4,LAT,LON,NOTES
0,FS21_FS9989,icraf-isric-soil-database,pervious,bare,soil,measured,-15.229167,23.266667,none
1,mobrmg.005-,asd,impervious,urban,other,measured,34.435800,-119.880400,urban reflectance spectra from santa barbara
2,v-LAI-6.7-LMA-0.006-CHL-58.8-N-2.2,prosail,pervious,vegetation,canopy,simulated,NaN,NaN,modeled canopy reflectance


Spectra can be subsampled to a random selection of spectra

In [7]:
sampled = library.subsample(n=10)

print(f"Array shape: {sampled.data.shape}")
print(f"Data type: {sampled.data.dtype}")
sampled.metadata.head(3)

Array shape: (10, 180)
Data type: float32


,NAME,SOURCE,LEVEL_1,LEVEL_2,LEVEL_3,LEVEL_4,LAT,LON,NOTES
0,v-LAI-2.7-LMA-0.007-CHL-10.3-N-1.8,prosail,pervious,vegetation,canopy,simulated,NaN,NaN,modeled canopy reflectance
1,FS21_FS372,icraf-isric-soil-database,pervious,bare,soil,measured,-3.820278,-73.3175,none
2,mozmom.001-,asd,impervious,urban,manhole,measured,34.446000,-119.8319,urban reflectance spectra from santa barbara


Spectra can also be subsampled randomly by class. you can pass different land cover classes (L1-L4) to the `by_type` parameter

In [8]:
sampled = library.subsample(n=5, by_type="vegetation")

print(f"Array shape: {sampled.data.shape}")
sampled.metadata.head(3)

Array shape: (5, 180)


,NAME,SOURCE,LEVEL_1,LEVEL_2,LEVEL_3,LEVEL_4,LAT,LON,NOTES
0,v-LAI-6.8-LMA-0.013-CHL-50.4-N-2.3,prosail,pervious,vegetation,canopy,simulated,NaN,NaN,modeled canopy reflectance
1,v-LAI-1.0-LMA-0.012-CHL-17.6-N-2.2,prosail,pervious,vegetation,canopy,simulated,NaN,NaN,modeled canopy reflectance
2,v-LAI-5.0-LMA-0.010-CHL-41.3-N-2.1,prosail,pervious,vegetation,canopy,simulated,NaN,NaN,modeled canopy reflectance
